# 05 · Audit: trying to break the results
**Brief: "What we look for".** Each section targets one trap (luck, leakage, mode changes, clustered events)
and ends with whether the shortlist survives it. It audits the detectors that notebook 04 kept, so **run
notebook 04 first**: this notebook reads `outputs/04_test_and_select.json` and computes the audits live.

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| Trap: testing many detectors and keeping the lucky ones | section 1 |
| Trap: leakage | sections 2 and 3 |
| Trap: machine mode changes | sections 4 (time drift) and 5 (model and age) |
| Trap: events that cluster together | section 5 |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import json, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- Code used in this notebook (from ../src) --------------------------------------------
import numpy as np
from data import Config, load_panel, dev_holdout_split   # src/data.py §1-§2: settings, data, dev/holdout machines
from detectors import FeatureBank                        # src/detectors.py §4: features, cached
from evaluation import (                                 # src/evaluation.py
    audits, MAPPING,                                     #   §7 the audits below; the part pairing they use
    load_results, check_results, save_results,           #   §9 read notebook 04's results; save this one's
)

> **Relies on:** `outputs/04_test_and_select.json`, saved by notebook 04 (the kept detectors, their rules, and the
> part pairing). The check below says whether it is up to date. Everything else is computed here.

In [3]:
display(check_results(["04_test_and_select"]))   # when notebook 04 ran, and whether it is still up to date
R = load_results("04_test_and_select")             # what notebook 04 found, for all five event types
cfg = Config(**{**R["config"], "percentiles": tuple(R["config"]["percentiles"])})   # the same settings 04 used
panel = load_panel()
bank = FeatureBank(panel)
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)
assert (dev + 1).tolist() == R["dev_machines"], "different dev/holdout split from notebook 04"
MAPPING.clear(); MAPPING.update(R["mapping"])      # each part's own signal and error codes, as 04 learned them
print("kept detectors (any failure):", R["kinds"]["any"]["kept"])

# The audits, computed live for the detectors 04 kept (~20 s): time drift, recovery guardrail, machine models
# and ages, leakage through the event hour, and 300 pure-noise detectors.
A = audits(panel, bank, cfg.with_(event="any"), R["kinds"]["any"], dev, hold, np.random.default_rng(cfg.seed))

,ran_at,H,W,status
notebook,,,,
04_test_and_select,2026-11-02 03:33:40,0,48,up to date


kept detectors (any failure): ['C09', 'C08', 'C10']


## 1 · Trap: keeping the lucky ones
Two checks:
* **Circular-shift null:** rotate each machine's alerts in time by a random offset (≥ 1 week), 500 times.
  This keeps the detector's firing rate and burstiness but breaks any real link to failures.
  A real precursor scores far above every rotation.
* **Noise detectors:** build 300 detectors on pure random noise, tune them exactly like ours, and keep the
  best one on dev. Its dev score shows how much "skill" selection alone can manufacture.

In [4]:
# For each kept detector: its real dev F1 next to the null distribution of rotated (shuffled-in-time) copies.
# null_mean / null_p95 = typical and 95th-percentile F1 that luck alone produces; z = how many null standard
# deviations the real score sits above the null mean. Anything above ~3 is far beyond luck.
print(pd.DataFrame([dict(id=r["cid"], dev_f1=r["dev"]["f1"], null_mean=r["null_f1_mean"],
                         null_p95=r["null_f1_p95"], z=r["null_z"])
                    for r in R["kinds"]["any"]["concepts"] if r["status"] == "kept"]))

# 300 detectors built on pure random noise, tuned the same way. The best one on dev shows how much "skill"
# selection alone can manufacture; its holdout score shows that skill disappearing on unseen machines.
lk = A["lucky"]
print(f"\n{len(lk['dev_f1'])} pure-noise detectors: best dev F1 {lk['best_dev']}, "
      f"the same detector on holdout {lk['best_hold']}")

    id  dev_f1  null_mean  null_p95      z
0  C08  0.7767     0.0576    0.0826  53.29
1  C09  0.9454     0.0500    0.0807  52.34
2  C10  0.5584     0.0668    0.0849  42.27

300 pure-noise detectors: best dev F1 0.0861, the same detector on holdout 0.071


**Conclusion.** Not luck. Each kept detector scores far above its null (z ≈ 42–53), where luck alone reaches
F1 ≈ 0.08. The best of 300 pure-noise detectors gets 0.09 on dev and 0.07 on holdout: selection by itself
manufactures almost nothing.

## 2 · Trap: leakage through the event hour
Every failure is logged together with its repair (a maintenance record at the same hour). If the scored
window included the failure hour itself, "a maintenance record exists" would look like a perfect precursor.
Our windows never include lag 0. Here we deliberately include it to show the trap.

In [5]:
# Same trivial "detector" (maintenance logged this hour), scored with and without the event hour.
# Expect: ~100% coverage and an absurd lift when the event hour is scored; 0 when it is not.
pd.DataFrame(A["leakage"])[["score_event_hour", "coverage", "precision", "lift"]]

,score_event_hour,coverage,precision,lift
0,False,0.0,NaN,0.0000
1,True,1.0,0.0,66963.1667


**Conclusion.** Scoring the failure hour gives a meaningless detector 100% coverage and a lift of ~67,000. Our
windows never score that hour, and the same detector then scores 0.

## 3 · Trap: counters that leak the future
"Hours since the target component's last maintenance" passes the null, lift and coverage gates with
lift ~1,800. But "normal operation" excludes the hours before failures, which is exactly where the counter
peaks. So its "normal" P99.99 quietly encodes where failures are. Fitting the threshold on Jan–Jun only
and scoring Jul–Dec exposes it. That's gate 4.

In [6]:
rows = []
for e, k in R["kinds"].items():
    for r in k["concepts"]:
        if r["cid"] == "C07":   # the maintenance-counter idea
            rows.append(dict(event=e, id=r["cid"], dev_lift=r["dev"]["lift"], hold_f1=r["hold"]["f1"],
                             causal_ref=r["causal_f1_ref"],   # Jul-Dec F1 with full-year thresholds
                             causal=r["causal_f1"],           # Jul-Dec F1 with Jan-Jun-only thresholds
                             status=r["status"], reason=r["reason"][:90]))
pd.DataFrame(rows)   # look for: huge dev_lift, then causal << causal_ref -> rejected

,event,id,dev_lift,hold_f1,causal_ref,causal,status,reason
0,any,C07,627.3877,0.1058,0.0468,0.0000,rejected,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.05 with full-year thresholds): i
1,comp1,C07,1758.2057,0.3077,0.1379,0.0000,rejected,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.14 with full-year thresholds): i
2,comp2,C07,1476.5329,0.2747,0.1791,0.1609,rejected,"passed all gates but outranked by three stronger, non-redundant detectors"
3,comp3,C07,1797.3222,0.2909,0.0541,0.0000,rejected,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.05 with full-year thresholds): i
4,comp4,C07,1982.2950,0.3433,0.2353,0.0000,rejected,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.24 with full-year thresholds): i


**Conclusion.** The counter (C07) shows huge lifts (600–2,000), then drops to F1 ≈ 0 when its threshold may use
only past data, for almost every event type. The causal gate rejects it. The kept detectors pass the same test
(section 4).

## 4 · Traps: time drift and the recovery guardrail
* **Time drift:** thresholds fitted on Jan–Jun only should work on Jul–Dec as well as full-year thresholds.
* **Recovery guardrail:** results should not hinge on the exact number of hours muted after a repair.

In [7]:
# Temporal: F1 on Jul-Dec with full-year thresholds vs. Jan-Jun-only thresholds (should be close).
display(pd.json_normalize(A["temporal"]))

# Recovery guardrail: R = hours after a failure/maintenance excluded from "normal" and muted for alerts.
pd.DataFrame(A["recovery"]).pivot(index="R", columns="cid", values=["precision", "coverage"]).round(3)

,cid,full_year_thresholds.lift,full_year_thresholds.precision,full_year_thresholds.coverage,full_year_thresholds.f1,full_year_thresholds.base_precision,full_year_thresholds.alerts_per_machine_month,full_year_thresholds.n_events,full_year_thresholds.n_alerts,first_half_thresholds.lift,first_half_thresholds.precision,first_half_thresholds.coverage,first_half_thresholds.f1,first_half_thresholds.base_precision,first_half_thresholds.alerts_per_machine_month,first_half_thresholds.n_events,first_half_thresholds.n_alerts
0,C08,34.1415,0.6059,0.9820,0.7494,0.0395,2.1666,334.0,1223.0,33.7973,0.6110,0.9850,0.7542,0.0395,2.1949,334.0,1239.0
1,C09,279.1724,0.9062,0.9341,0.9200,0.0395,1.4172,334.0,800.0,220.5628,0.8774,0.8683,0.8728,0.0395,1.3003,334.0,734.0
2,C10,16.8336,0.3995,0.8832,0.5501,0.0395,3.9647,334.0,2238.0,17.9784,0.4214,0.8862,0.5712,0.0395,3.7964,334.0,2143.0


precision               coverage              
cid       C08    C09    C10      C08    C09    C10
R                                                 
0       0.592  0.920  0.395    0.977  0.909  0.903
6       0.602  0.921  0.400    0.977  0.909  0.903
24      0.628  0.927  0.413    0.977  0.909  0.903
72      0.686  0.941  0.452    0.977  0.909  0.903

**Conclusion.** Little time drift: with Jan–Jun-only thresholds, Jul–Dec F1 is 0.87 / 0.75 / 0.57 (C09 / C08 / C10),
against 0.92 / 0.75 / 0.55 with full-year thresholds. C09 loses the most (0.05), because its per-machine P99
line then rests on half a year of readings; C08 and C10 don't lose anything. The recovery guardrail barely
matters: muting 0 to 72 h after a repair moves C09's precision only from 0.92 to 0.94, with coverage 0.91 throughout.

## 5 · Traps: machine mode changes and clustered events
* **Machine mode changes:** each kept detector should work about equally well on every machine model and age band.
* **Clustered events:** too rare to test here. Only 3 failures in the whole year follow another within 7 days
  (notebook 01, section 4). Every interval is still a cluster bootstrap over machines, so events on the same
  machine are never treated as independent.

In [8]:
# Machine mode changes: does each kept detector work equally well on every machine model and age band?
# (all machines; thresholds fitted on dev). Similar values across rows = no regime is treated differently.
reg = pd.DataFrame(A["regimes"])[["cid", "kind", "group", "coverage", "precision"]]
display(reg.pivot_table(index=["kind", "group"], columns="cid", values=["coverage", "precision"]).round(3))

coverage               precision              
cid                   C08    C09    C10       C08    C09    C10
kind  group                                                    
age   age 0-5y      0.974  0.974  0.879     0.552  0.923  0.361
      age 12-15y    0.994  0.929  0.885     0.673  0.906  0.413
      age 16-20y    0.980  0.938  0.938     0.691  0.945  0.472
      age 6-11y     0.972  0.944  0.895     0.584  0.896  0.354
model model1        0.976  0.922  0.940     0.687  0.940  0.457
      model2        0.987  0.962  0.942     0.644  0.928  0.449
      model3        0.990  0.942  0.860     0.626  0.934  0.370
      model4        0.970  0.947  0.894     0.606  0.884  0.397

**Conclusion.** No model or age band is treated differently. C09 keeps precision 0.88–0.94 and coverage
0.92–0.97 in every group. C08 and C10 vary more (C08 precision 0.55–0.69) but stay far above the 4% that a
random alarm gets. Clustered events can't distort this: only 3 follow another within a week.

## Notebook summary
| Trap | Result |
|---|---|
| Lucky picks from many tries | Kept detectors sit far above a shuffled-timing null (z ≈ 42–53). The best of 300 pure-noise detectors scores F1 ≈ 0.09 on dev and 0.07 on holdout. |
| Leakage through the event hour | Scoring the failure hour turns "a repair was logged" into a perfect fake precursor. Our windows never include it. |
| Counters that leak the future | "Hours since maintenance" passes every statistical gate, then collapses with past-only thresholds. The causal gate rejects it. |
| Time drift | Thresholds fitted on Jan–Jun alone work almost as well on Jul–Dec (C09 F1 0.87 vs 0.92; C08 and C10 unchanged). |
| Machine mode changes | C09 is stable across all machine models and age bands (precision 0.88–0.94, coverage 0.92–0.97). C08 and C10 vary more (e.g. C08 precision 0.55–0.69), but stay far above the 4% random-alarm level. |
| Clustered events | Too rare to matter here: 3 failures in the whole year follow another within 7 days. Intervals come from a cluster bootstrap over machines anyway. |

## 6 · Save this notebook's results
The audit results are saved to `outputs/05_audit.json` for notebook 06.

In [9]:
save_results("05_audit", dict(audits=A), cfg)   # src/evaluation.py §9

saved outputs/05_audit.json


PosixPath('/Users/avijay/Documents/oden_precursors/outputs/05_audit.json')